In [1]:
import os
%pwd

'd:\\Github Projects\\NLP MLOPs Project\\research'

In [2]:
os.chdir("../")

In [3]:
%pwd

'd:\\Github Projects\\NLP MLOPs Project'

Config Entity

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: Path
    local_data_file: Path
    unzip_dir: Path

Configuration Manger

In [12]:
from src.textSummarizer.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.textSummarizer.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self,
                config_path = CONFIG_FILE_PATH,
                params_filepath = PARAMS_FILE_PATH
                ):
            self.config = read_yaml(config_path)
            self.params = read_yaml(params_filepath)

            create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config =  DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config


Components

In [15]:
import os
import urllib.request as request
import zipfile
from src.textSummarizer.logging import logger


class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            # Implementation for downloading file
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"File downloaded successfully to {self.config.local_data_file}")
        else:
            logger.info(f"File already exists at {self.config.local_data_file}")

    def extract_zip_file(self):
        if not os.path.exists(self.config.unzip_dir):
            create_directories([self.config.unzip_dir])
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)

In [16]:
config = ConfigurationManager()

data_ingestion_config = config.get_data_ingestion_config()
data_ingestion = DataIngestion(config = data_ingestion_config)

data_ingestion.download_file()
data_ingestion.extract_zip_file()

[2026-09-06 11:36:03,510: INFO : common : yaml file: config\config.yaml loaded successfully]
[2026-09-06 11:36:03,516: INFO : common : yaml file: params.yaml loaded successfully]
[2026-09-06 11:36:03,520: INFO : common : created directory at: artifacts]
[2026-09-06 11:36:03,524: INFO : common : created directory at: artifacts/data_ingestion]
[2026-09-06 11:36:26,545: INFO : 2781883786 : File downloaded successfully to artifacts/data_ingestion/data.zip]
